# 07. 파이썬 기초 - Selenium 브라우저 자동화

`requests` 는 JavaScript 로 나중에 그려지는 내용(무한 스크롤, 로그인 후 화면 등)을 가져오지 못합니다.
**Selenium** 은 실제 크롬 브라우저를 프로그램으로 조종해 이런 동적 페이지를 다룹니다.
이 프로젝트의 `08.Selenium사용_*.ipynb`, `selenium_infinite_scroll.py` 가 그 예입니다.

**다루는 내용**
1. 드라이버 생성과 페이지 열기
2. 요소 찾기 (find_element / By)
3. 클릭과 텍스트 입력
4. 대기 (implicit / explicit)
5. 스크린샷과 종료
6. 예외처리

> ⚠️ 이 노트북은 **실제 크롬 브라우저** 가 필요해 자동 실행하지 않습니다.
> 크롬이 설치된 환경에서 셀을 직접 실행해 보세요.
> 설치: `pip install selenium` (Selenium 4.6+ 는 드라이버 자동 관리)

## 1. 드라이버 생성과 페이지 열기

`webdriver.Chrome()` 으로 브라우저를 켜고 `.get(url)` 로 페이지를 엽니다.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
# options.add_argument('--headless=new')  # 창 없이 백그라운드 실행하려면 주석 해제
options.add_argument('--window-size=1200,800')

# Selenium 4.6+ 는 chromedriver 를 자동으로 내려받아 관리한다
driver = webdriver.Chrome(options=options)

# 연습용 사이트 열기
driver.get('https://quotes.toscrape.com/')
print('페이지 제목:', driver.title)

## 2. 요소 찾기 — find_element / find_elements

- `find_element(By.XxX, '값')` : 조건에 맞는 **첫 번째** 요소 하나
- `find_elements(...)` : 조건에 맞는 **모든** 요소 → 리스트 (없으면 빈 리스트)

`By` 로 찾는 방식을 지정합니다 (CSS 선택자, id, class 등).

> ⚠️ Selenium 4 에서는 `find_element_by_xxx()` 같은 옛 메서드가 **삭제**되었습니다.
> 반드시 `find_element(By.XXX, ...)` 형태를 사용하세요.

In [ ]:
from selenium.webdriver.common.by import By

# CSS 선택자로 명언 카드 여러 개 찾기 → 리스트
quotes = driver.find_elements(By.CSS_SELECTOR, 'div.quote')
print('찾은 명언 수:', len(quotes))

# 첫 카드에서 텍스트/작가 추출 (find_element 는 요소 안에서도 사용 가능)
first = quotes[0]
text = first.find_element(By.CSS_SELECTOR, 'span.text').text
author = first.find_element(By.CSS_SELECTOR, 'small.author').text
print('명언:', text)
print('작가:', author)

### By 의 주요 종류

| 방식 | 예시 | 설명 |
|------|------|------|
| `By.CSS_SELECTOR` | `'div.quote'` | CSS 선택자 (가장 많이 사용) |
| `By.ID` | `'login_field'` | id 속성 |
| `By.CLASS_NAME` | `'quote'` | class 속성 |
| `By.NAME` | `'commit'` | name 속성 |
| `By.TAG_NAME` | `'a'` | 태그 이름 |
| `By.XPATH` | `'//div[@class]'` | XPath 경로 |

## 3. 클릭과 텍스트 입력

- `.click()` : 버튼/링크 클릭
- `.send_keys('텍스트')` : 입력창에 타이핑
- `.send_keys(Keys.RETURN)` : 엔터 키

In [ ]:
from selenium.webdriver.common.keys import Keys

# 로그인 페이지로 이동 (연습용)
driver.get('https://quotes.toscrape.com/login')

# 아이디/비밀번호 입력창 찾아서 입력
driver.find_element(By.CSS_SELECTOR, 'input#username').send_keys('test')
driver.find_element(By.CSS_SELECTOR, 'input#password').send_keys('secret')

# 비밀번호 칸에서 엔터로 제출하거나, 로그인 버튼 클릭
driver.find_element(By.CSS_SELECTOR, 'input[type="submit"]').click()
print('현재 URL:', driver.current_url)

## 4. 대기(wait) — 매우 중요 

페이지 요소는 **바로 나타나지 않을 수 있습니다.** 로딩 전에 찾으면 오류가 납니다.

- **암묵적 대기(implicit)**: 모든 요소 탐색에 '최대 N초 기다림' 을 한 번만 설정
- **명시적 대기(explicit)**: 특정 조건이 만족될 때까지 기다림 (권장)

> 두 방식을 **섞어 쓰지 마세요.** 대기 시간이 예측 불가능해집니다.

In [ ]:
# 방법 A) 암묵적 대기 — 간단하지만 조건 지정 불가
driver.implicitly_wait(5)   # 요소를 찾을 때 최대 5초까지 기다림

# 방법 B) 명시적 대기 — '이 조건이 될 때까지' 기다림 (실무 권장)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver.get('https://quotes.toscrape.com/')
wait = WebDriverWait(driver, 10)   # 최대 10초

# 'div.quote 요소가 나타나면(clickable) 진행' — 나타나는 즉시 다음 줄 실행
first_quote = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, 'div.quote'))
)
print('명언이 로딩됨:', first_quote is not None)

## 5. 스크린샷과 종료

- `.save_screenshot(경로)` : 현재 화면을 이미지로 저장 (디버깅에 유용)
- `.quit()` : 브라우저 종료 (**반드시 호출**해서 자원 정리)

In [ ]:
import os
os.makedirs('img', exist_ok=True)

# 현재 화면을 이미지로 저장
driver.save_screenshot('img/quotes.png')
print('스크린샷 저장 완료')

# 브라우저 종료
driver.quit()
print('브라우저 종료')

## 6. 예외처리 — 안정적인 자동화

요소를 못 찾거나 시간이 초과되면 예외가 납니다.
03편의 `try/except/finally` 로 감싸고, **finally 에서 반드시 quit()** 합니다.
(예외가 나도 브라우저가 닫히도록)

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd

driver = webdriver.Chrome()
try:
    driver.get('https://quotes.toscrape.com/')

    # 명언들을 '딕셔너리들의 리스트' 로 수집 (01~05편 종합)
    rows = []
    for card in driver.find_elements(By.CSS_SELECTOR, 'div.quote'):
        rows.append({
            'text': card.find_element(By.CSS_SELECTOR, 'span.text').text,
            'author': card.find_element(By.CSS_SELECTOR, 'small.author').text,
        })

    df = pd.DataFrame(rows)   # → DataFrame (04편)
    print(df)

except (NoSuchElementException, TimeoutException) as e:
    print('요소를 찾지 못했습니다:', e)
finally:
    driver.quit()   # 성공/실패와 무관하게 항상 종료
    print('정리 완료')

## 정리
- **driver = webdriver.Chrome()** → **driver.get(url)** 로 시작
- **find_element(s)(By.CSS_SELECTOR, ...)** 로 요소 찾기 (옛 `find_element_by_*` 는 삭제됨)
- **.click() / .send_keys()** 로 조작, **Keys.RETURN** 으로 엔터
- **WebDriverWait + expected_conditions** 로 명시적 대기 (권장)
- **try/except/finally** 로 감싸고 **finally 에서 quit()**
